<a href="https://colab.research.google.com/github/Sarah-Manzoor-158/Machine-learning-course/blob/main/puzzle_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Artificial Intelligence Assignemnt Puzzle game

Imports & Puzzle Mechanics

In [1]:
import heapq
import random
import time
from collections import deque
import matplotlib.pyplot as plt

# Goal state defined in the assignment (0 represents the blank space)
GOAL = (1, 2, 3, 8, 0, 4, 7, 6, 5)

# Precompute goal (row, col) coordinates for each tile 1-8 for fast heuristic lookups
GOAL_POS = {tile: (idx // 3, idx % 3) for idx, tile in enumerate(GOAL) if tile != 0}


def print_board(state):
    """Prints a 9-element tuple cleanly as a 3x3 grid."""
    for i in range(0, 9, 3):
        row = [" " if x == 0 else str(x) for x in state[i : i + 3]]
        print(" ".join(row))
    print()


def is_goal(state):
    """Checks if the current board matches the goal configuration."""
    return state == GOAL


def successors(state):
    """Returns a list of (move_name, new_state) for all legal blank slides."""
    blank_idx = state.index(0)
    row, col = blank_idx // 3, blank_idx % 3
    results = []

    # Possible directions: (Move Name, row_change, col_change)
    directions = [("Up", -1, 0), ("Down", 1, 0), ("Left", 0, -1), ("Right", 0, 1)]

    for move_name, dr, dc in directions:
        new_r, new_c = row + dr, col + dc
        if 0 <= new_r < 3 and 0 <= new_c < 3:
            new_idx = new_r * 3 + new_c
            board_list = list(state)
            board_list[blank_idx], board_list[new_idx] = (
                board_list[new_idx],
                board_list[blank_idx],
            )
            results.append((move_name, tuple(board_list)))

    return results


def shuffle(n, seed=None):
    """Creates a guaranteed solvable puzzle by making n random moves from GOAL."""
    rng = random.Random(seed)
    current = GOAL
    prev_state = None

    for _ in range(n):
        valid_moves = successors(current)
        candidates = [s for _, s in valid_moves if s != prev_state]
        prev_state = current
        current = rng.choice(candidates)

    return current


def verify_solution(start, moves):
    """Replays a list of moves from start to confirm it reaches GOAL."""
    current = start
    for move in moves:
        next_states = dict(successors(current))
        if move not in next_states:
            return False
        current = next_states[move]
    return is_goal(current)

print("Cell 1 Ready: Puzzle functions loaded!")

Cell 1 Ready: Puzzle functions loaded!


Uninformed Search Algorithms

In [2]:
def bfs(start):
    """Breadth-First Search using a FIFO queue."""
    start_time = time.perf_counter()
    if is_goal(start):
        return [], 0, time.perf_counter() - start_time

    queue = deque([(start, [])])
    visited = {start}
    nodes_expanded = 0

    while queue:
        state, path = queue.popleft()
        nodes_expanded += 1

        for move, next_state in successors(state):
            if next_state not in visited:
                new_path = path + [move]
                if is_goal(next_state):
                    return new_path, nodes_expanded, time.perf_counter() - start_time
                visited.add(next_state)
                queue.append((next_state, new_path))

    return None, nodes_expanded, time.perf_counter() - start_time


def dfs(start, limit=30):
    """Depth-Limited DFS using a LIFO stack."""
    start_time = time.perf_counter()
    stack = [(start, [])]
    visited_depth = {start: 0}
    nodes_expanded = 0

    while stack:
        state, path = stack.pop()
        if is_goal(state):
            return path, nodes_expanded, time.perf_counter() - start_time

        if len(path) < limit:
            nodes_expanded += 1
            for move, next_state in reversed(successors(state)):
                new_depth = len(path) + 1
                if next_state not in visited_depth or new_depth < visited_depth[next_state]:
                    visited_depth[next_state] = new_depth
                    stack.append((next_state, path + [move]))

    return None, nodes_expanded, time.perf_counter() - start_time


def ucs(start):
    """Uniform Cost Search ordered by path cost g(n)."""
    start_time = time.perf_counter()
    counter = 0
    heap = [(0, counter, start, [])]
    best_g = {start: 0}
    nodes_expanded = 0

    while heap:
        g, _, state, path = heapq.heappop(heap)
        if g > best_g.get(state, float("inf")):
            continue

        if is_goal(state):
            return path, nodes_expanded, time.perf_counter() - start_time

        nodes_expanded += 1
        for move, next_state in successors(state):
            new_g = g + 1
            if new_g < best_g.get(next_state, float("inf")):
                best_g[next_state] = new_g
                counter += 1
                heapq.heappush(heap, (new_g, counter, next_state, path + [move]))

    return None, nodes_expanded, time.perf_counter() - start_time

print("Cell 2 Ready: BFS, DFS, and UCS loaded!")

Cell 2 Ready: BFS, DFS, and UCS loaded!


Heuristics  h1, h2 ,h3

In [3]:
def h_misplaced(state):
    """h1: Counts how many tiles (excluding blank) are not in their goal square."""
    return sum(
        1 for idx, tile in enumerate(state) if tile != 0 and tile != GOAL[idx]
    )


def h_manhattan(state):
    """h2: Sum of Manhattan distances (row diff + col diff) for all tiles."""
    distance = 0
    for idx, tile in enumerate(state):
        if tile != 0:
            curr_r, curr_c = idx // 3, idx % 3
            goal_r, goal_c = GOAL_POS[tile]
            distance += abs(curr_r - goal_r) + abs(curr_c - goal_c)
    return distance


def h_linear_conflict(state):
    """
    Bonus h3: Manhattan distance + 2 for every pair of tiles in the same row or column
    that belong in that row/column, but are reversed relative to their goal order.
    """
    dist = h_manhattan(state)
    conflicts = 0

    # Check row conflicts
    for r in range(3):
        row_tiles = []
        for c in range(3):
            tile = state[r * 3 + c]
            if tile != 0 and GOAL_POS[tile][0] == r:
                row_tiles.append((c, GOAL_POS[tile][1]))
        for i in range(len(row_tiles)):
            for j in range(i + 1, len(row_tiles)):
                if row_tiles[i][1] > row_tiles[j][1]:
                    conflicts += 1

    # Check column conflicts
    for c in range(3):
        col_tiles = []
        for r in range(3):
            tile = state[r * 3 + c]
            if tile != 0 and GOAL_POS[tile][1] == c:
                col_tiles.append((r, GOAL_POS[tile][0]))
        for i in range(len(col_tiles)):
            for j in range(i + 1, len(col_tiles)):
                if col_tiles[i][1] > col_tiles[j][1]:
                    conflicts += 1

    return dist + 2 * conflicts

print("Cell 3 Ready: Heuristics (h1, h2, h3) loaded!")

Cell 3 Ready: Heuristics (h1, h2, h3) loaded!


Informed Search Algorithms — Greedy & A*

In [4]:
def greedy(start, h):
    """Greedy Best-First Search ordered strictly by heuristic h(n)."""
    start_time = time.perf_counter()
    counter = 0
    heap = [(h(start), counter, start, [])]
    visited = {start}
    nodes_expanded = 0

    while heap:
        _, _, state, path = heapq.heappop(heap)
        if is_goal(state):
            return path, nodes_expanded, time.perf_counter() - start_time

        nodes_expanded += 1
        for move, next_state in successors(state):
            if next_state not in visited:
                visited.add(next_state)
                counter += 1
                heapq.heappush(heap, (h(next_state), counter, next_state, path + [move]))

    return None, nodes_expanded, time.perf_counter() - start_time


def astar(start, h):
    """A* Search ordered by f(n) = g(n) + h(n)."""
    start_time = time.perf_counter()
    counter = 0
    heap = [(h(start), counter, 0, start, [])]
    best_g = {start: 0}
    nodes_expanded = 0

    while heap:
        f, _, g, state, path = heapq.heappop(heap)
        if g > best_g.get(state, float("inf")):
            continue

        if is_goal(state):
            return path, nodes_expanded, time.perf_counter() - start_time

        nodes_expanded += 1
        for move, next_state in successors(state):
            new_g = g + 1
            if new_g < best_g.get(next_state, float("inf")):
                best_g[next_state] = new_g
                counter += 1
                new_f = new_g + h(next_state)
                heapq.heappush(heap, (new_f, counter, new_g, next_state, path + [move]))

    return None, nodes_expanded, time.perf_counter() - start_time

print("Cell 4 Ready: Greedy and A* loaded!")

Cell 4 Ready: Greedy and A* loaded!
